# Module 06 — Feature Pyramid Networks (SOLUTIONS)

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
import matplotlib.pyplot as plt
import numpy as np
from fpn import FPN

# Load backbone and FPN
backbone = models.resnet50(weights='IMAGENET1K_V1').eval()
feature_maps = {}
for name in ['layer1','layer2','layer3','layer4']:
    getattr(backbone, name).register_forward_hook(
        lambda m, i, o, n=name: feature_maps.__setitem__(n, o))

tfm = T.Compose([T.Resize(512), T.CenterCrop(512), T.ToTensor(),
                  T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

from PIL import Image
import urllib.request, io
URL = 'https://upload.wikimedia.org/wikipedia/commons/thumb/a/ab/Abraham_Lincoln_O-77_matte_collodion_print.jpg/400px-Abraham_Lincoln_O-77_matte_collodion_print.jpg'
try:
    with urllib.request.urlopen(URL) as r: data = r.read()
    img = Image.open(io.BytesIO(data)).convert('RGB')
except:
    img = Image.fromarray(np.random.randint(0,200,(512,512,3), dtype=np.uint8))

x = tfm(img).unsqueeze(0)
with torch.no_grad(): backbone(x)

in_channels = [feature_maps[k].shape[1] for k in ['layer1','layer2','layer3','layer4']]
fpn = FPN(in_channels, 256)
feat_dict = {str(i): feature_maps[k] for i,k in enumerate(['layer1','layer2','layer3','layer4'])}
with torch.no_grad():
    pyramid = fpn(feat_dict)

# Visualize
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes[0,0].imshow(img); axes[0,0].set_title('Input image'); axes[0,0].axis('off')

for ax, (name, feat) in zip(axes.flat[1:], pyramid.items()):
    act = feat.abs().mean(dim=1).squeeze().detach().numpy()
    im = ax.imshow(act, cmap='hot')
    ax.set_title(f'{name}: {tuple(feat.shape[2:])}')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('FPN Feature Map Activations', fontsize=14)
plt.tight_layout(); plt.show()